# Tes 3 

Sama kyak tes 2 masih pake lightihng normalized (tetep ngefek cahaya di segmen), konversi warna juga, ektraksi gambar bedanya ini di segmentasi (grabcut / threshold) bukan dari 1 gambar utuh

konfigurasi dll

In [4]:
if 'CFG' not in globals():
    CFG = {
        "img_size": (256,256),
        "glcm_size": (128,128),
        "hsv_bins": (8,8,4),
        "glcm_levels": 8,
        "glcm_distances": [1],
        "glcm_angles": [0, np.pi/4, np.pi/2, 3*np.pi/4],
        "random_state": 42,
    }
np.random.seed(CFG["random_state"])

In [5]:
base = Path(CFG["base_dir"])
df =  pd.read_excel("data_original_edit.xlsx")

# Normalisasi nama kolom (kadang beda kapital/spasi)
df.columns = [c.strip() for c in df.columns]

needed = {"Image Before Eaten", "Image After Eaten", "Weight Before Eaten (g)", "Weight After Eaten (g)", "Name of the food"}
missing = needed - set(df.columns)
assert not missing, f"Kolom hilang: {missing}"

# Bentuk y: 100 * after/before 
wb = df["Weight Before Eaten (g)"].astype(float)
wa = df["Weight After Eaten (g)"].astype(float)
y = 100.0 * np.divide(wa, wb, out=np.zeros_like(wa, dtype=float), where=wb!=0)
y = np.clip(y, 0, 100)
df["y_percent_leftover"] = y

df["outlier_weight"] = (wa > wb) | (wb < 1)
df.head()

,ID,Name of the food,Image Before Eaten,Weight Before Eaten (g),Image After Eaten,Weight After Eaten (g),Visual Estimation by Observer (1-7),y_percent_leftover,outlier_weight
0,1,Bubur,000_000_DSC_0016_bef.JPG,343,000_000_DSC_0032_aft.JPG,330,1,96.209913,False
1,2,Nasi,001_001_DSC_0059_bef.JPG,130,001_001_DSC_0108_aft.JPG,122,3,93.846154,False
2,3,Nasi,001_002_DSC_0066_bef.JPG,135,001_002_DSC_0095_aft.JPG,127,1,94.074074,False
3,4,Nasi,001_003_DSC_0067_bef.JPG,146,001_003_DSC_0097_aft.JPG,1,7,0.684932,False
4,5,Nasi,001_008_DSC_0055_bef.JPG,139,001_008_DSC_0101_aft.JPG,25,6,17.985612,False


grouping biar 1 baris

In [6]:


# ====== Index file gambar & Group by session ======
def build_index(folder: Path):
    idx = {}
    for p in folder.rglob("*"):
        if p.is_file():
            idx[p.name.lower()] = p
    return idx

def to_key(s):
    return Path(str(s).strip()).name.lower()

before_root = base / CFG["before_dirname"]
after_root  = base / CFG["after_dirname"]
assert before_root.exists(), f"Folder tidak ditemukan: {before_root}"
assert after_root.exists(),  f"Folder tidak ditemukan: {after_root}"

before_index = build_index(before_root)
after_index  = build_index(after_root)
print("Index sizes -> before:", len(before_index), "| after:", len(after_index))

df["before_key"] = df["Image Before Eaten"].apply(to_key)
df["after_key"]  = df["Image After Eaten"].apply(to_key)
df["before_path"] = df["before_key"].map(before_index)
df["after_path"]  = df["after_key"].map(after_index)

df_ok = df[df["before_path"].notna() & df["after_path"].notna()].reset_index(drop=True)

def infer_group(fname: str, fallback: str) -> str:
    m = re.match(r"^(\d{3}_\d{3})", fname)
    if m: return m.group(1)
    return f"FOOD::{fallback.strip().lower()}"

df_ok["group"] = [
    infer_group(Path(b).name, food)
    for b, food in zip(df_ok["Image Before Eaten"].astype(str), df_ok["Name of the food"].astype(str))
]

print("Rows valid:", len(df_ok), "Contoh kolom kunci:")
df_ok[["Image Before Eaten","Image After Eaten","group"]].head()



Index sizes -> before: 524 | after: 524
Rows valid: 524 Contoh kolom kunci:


,Image Before Eaten,Image After Eaten,group
0,000_000_DSC_0016_bef.JPG,000_000_DSC_0032_aft.JPG,000_000
1,001_001_DSC_0059_bef.JPG,001_001_DSC_0108_aft.JPG,001_001
2,001_002_DSC_0066_bef.JPG,001_002_DSC_0095_aft.JPG,001_002
3,001_003_DSC_0067_bef.JPG,001_003_DSC_0097_aft.JPG,001_003
4,001_008_DSC_0055_bef.JPG,001_008_DSC_0101_aft.JPG,001_008


kode di tes 2 yang normalized ligthing

In [7]:
def normalize_lighting(img_rgb, mode="clahe"):
    # fallback jika belum ada dari Tes 2
    if mode == "grayworld":
        mean_rgb = np.mean(img_rgb, axis=(0,1))
        gray_mean = np.mean(mean_rgb)
        scale = gray_mean / (mean_rgb + 1e-8)
        img_rgb = np.clip(img_rgb * scale, 0, 255).astype(np.uint8)
    else:
        lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        l2 = clahe.apply(l)
        img_rgb = cv2.cvtColor(cv2.merge([l2,a,b]), cv2.COLOR_LAB2RGB)
    return img_rgb

def load_img_norm(path: Path, size=(256,256), mode="clahe"):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None: return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, size, interpolation=cv2.INTER_AREA)
    return normalize_lighting(img, mode=mode)


segmentasi / masking

In [10]:
def grabcut_center_mask(img_rgb, rect_scale=0.75, iters=3):
    h, w = img_rgb.shape[:2]
    cx, cy = w//2, h//2
    rw, rh = int(w*rect_scale), int(h*rect_scale)
    x = max(0, cx - rw//2); y = max(0, cy - rh//2)
    rect = (x, y, min(rw, w-x), min(rh, h-y))

    bgdModel = np.zeros((1,65), np.float64)
    fgdModel = np.zeros((1,65), np.float64)
    mask = np.zeros((h, w), np.uint8)
    cv2.grabCut(img_rgb, mask, rect, bgdModel, fgdModel, iters, cv2.GC_INIT_WITH_RECT)
    mask_bin = np.where((mask==cv2.GC_FGD) | (mask==cv2.GC_PR_FGD), 1, 0).astype(np.uint8)

    # bersihkan noise kecil
    kernel = np.ones((3,3), np.uint8)
    mask_bin = cv2.morphologyEx(mask_bin, cv2.MORPH_OPEN, kernel, iterations=1)
    mask_bin = cv2.morphologyEx(mask_bin, cv2.MORPH_CLOSE, kernel, iterations=1)
    return mask_bin

def apply_mask(img_rgb, mask_bin):
    if mask_bin.ndim == 2:
        mask3 = np.repeat(mask_bin[..., None], 3, axis=2)
    else:
        mask3 = mask_bin
    out = img_rgb.copy()
    out[mask3==0] = 0
    return out


sama aja ekstraksi fitur tapi dipake setelah segmentasi

In [11]:
def hsv_hist(img_rgb, bins=(8,8,4)):
    hsv  = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    hist = cv2.calcHist([hsv],[0,1,2], None, bins, [0,180,0,256,0,256]).flatten()
    hist = hist / (hist.sum() + 1e-8)
    return hist.astype(np.float32)

def glcm_feats(img_rgb, size=(128,128), levels=8, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4]):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    gray = cv2.resize(gray, size, interpolation=cv2.INTER_AREA)
    q = np.floor((gray.astype(np.float32)/256.0)*levels).astype(np.uint8)
    g = graycomatrix(q, distances=distances, angles=angles, levels=levels, symmetric=True, normed=True)
    props = ["contrast","homogeneity","energy","correlation"]
    vals = [np.mean(graycoprops(g, p)) for p in props]
    return np.array(vals, dtype=np.float32)

def pair_metrics(img_bef, img_aft):
    g1 = cv2.cvtColor(img_bef, cv2.COLOR_RGB2GRAY)
    g2 = cv2.cvtColor(img_aft, cv2.COLOR_RGB2GRAY)
    ssim_val = ssim(g1, g2, data_range=max(1e-6, float(g2.max()-g2.min())))
    mse_val  = np.mean((g1.astype(np.float32)-g2.astype(np.float32))**2)
    return np.array([ssim_val, mse_val], dtype=np.float32)

def extract_features_pair_segmented(path_bef: Path, path_aft: Path):
    # 1) Load + lighting normalized
    img_b = load_img_norm(path_bef, CFG["img_size"], mode="clahe")
    img_a = load_img_norm(path_aft, CFG["img_size"], mode="clahe")
    if img_b is None or img_a is None:
        return None

    # 2) Segmentasi (mask makanan)
    mb = grabcut_center_mask(img_b, rect_scale=0.75, iters=3)
    ma = grabcut_center_mask(img_a, rect_scale=0.75, iters=3)

    # 3) Terapkan mask (nolkan background)
    img_bm = apply_mask(img_b, mb)
    img_am = apply_mask(img_a, ma)

    # 4) Fitur warna/tekstur pada area makanan
    h_b = hsv_hist(img_bm, CFG["hsv_bins"])
    h_a = hsv_hist(img_am, CFG["hsv_bins"])
    h_d = h_a - h_b

    g_b = glcm_feats(img_bm, CFG["glcm_size"], CFG["glcm_levels"], CFG["glcm_distances"], CFG["glcm_angles"])
    g_a = glcm_feats(img_am, CFG["glcm_size"], CFG["glcm_levels"], CFG["glcm_distances"], CFG["glcm_angles"])
    g_d = g_a - g_b

    pm  = pair_metrics(img_bm, img_am)  # ssim/mse di versi masked

    # 5) Fitur area (distandarkan / total piksel)
    area_b = float(mb.sum()) / (mb.shape[0]*mb.shape[1])
    area_a = float(ma.sum()) / (ma.shape[0]*ma.shape[1])
    delta_area = area_a - area_b
    ratio_area = area_a / (area_b + 1e-8)
    area_feats = np.array([area_b, area_a, delta_area, ratio_area], dtype=np.float32)

    # 6) Gabungkan
    feat = np.concatenate([h_b, h_a, h_d, g_b, g_a, g_d, pm, area_feats], axis=0)
    return feat


ekstraksi fitur semua data → Sabar sering lama

In [19]:
# Kumpulkan fitur untuk seluruh baris valid df_ok (sama seperti Tes 2)
features_B, valid_idx_B, skipped_B = [], [], []
for i, row in tqdm(df_ok.iterrows(), total=len(df_ok)):
    try:
        f = extract_features_pair_segmented(row["before_path"], row["after_path"])
    except Exception as e:
        f = None
        skipped_B.append((i,str(e)))
    if f is not None and np.all(np.isfinite(f)):
        features_B.append(f); valid_idx_B.append(i)

X_B = np.vstack(features_B)
y_B = df_ok.loc[valid_idx_B, "y_percent_leftover"].to_numpy().astype(np.float32)
groups_B = df_ok.loc[valid_idx_B, "group"].to_numpy()

print("X_B shape:", X_B.shape, "| y_B:", y_B.shape, "| groups_B:", groups_B.shape)
print("Skipped (segmentation errors):", len(skipped_B))

# Nama fitur lengkap (782 + 4 area = 786)
def build_feature_names_B(h_bins=(8,8,4)):
    H,S,V = h_bins
    names = []
    def add_hsv(prefix):
        for h in range(H):
            for s in range(S):
                for v in range(V):
                    names.append(f"{prefix}_hsv[{h},{s},{v}]")
    add_hsv("bef"); add_hsv("aft"); add_hsv("dlt")
    glcm_props = ["glcm_contrast","glcm_homogeneity","glcm_energy","glcm_correlation"]
    names += [f"bef_{p}" for p in glcm_props]
    names += [f"aft_{p}" for p in glcm_props]
    names += [f"dlt_{p}" for p in glcm_props]
    names += ["pair_ssim","pair_mse"]
    # fitur area
    names += ["area_before","area_after","delta_area","ratio_area"]
    return names

feature_names_B = build_feature_names_B(CFG["hsv_bins"])
assert X_B.shape[1] == len(feature_names_B), f"Fitur: {X_B.shape[1]} vs names: {len(feature_names_B)}"


100%|██████████| 524/524 [04:40<00:00,  1.87it/s]

X_B shape: (524, 786) | y_B: (524,) | groups_B: (524,)
Skipped (segmentation errors): 0


In [20]:
# Nested CV (outer 5 x inner 3) untuk pemilihan hyperparam yang adil
outer_cv = GroupKFold(n_splits=5)
inner_cv = GroupKFold(n_splits=3)
param_grid = {
    "n_estimators": [300, 600],
    "max_depth": [None, 12, 20],
    "max_features": ["sqrt", 0.5],
    "min_samples_leaf": [1, 3],
    "random_state": [CFG["random_state"]],
    "n_jobs": [-1],
}

outer_scores = []
fold = 1
for tr_idx, te_idx in outer_cv.split(X_B, y_B, groups_B):
    Xtr, Xte = X_B[tr_idx], X_B[te_idx]
    ytr, yte = y_B[tr_idx], y_B[te_idx]
    gtr      = groups_B[tr_idx]

    gs = GridSearchCV(RandomForestRegressor(), param_grid,
                      scoring="neg_mean_absolute_error",
                      cv=inner_cv.split(Xtr, ytr, gtr),
                      n_jobs=-1, verbose=0)
    gs.fit(Xtr, ytr)
    bestB = gs.best_estimator_
    yhat  = bestB.predict(Xte)

    mae = mean_absolute_error(yte, yhat)
    mse = mean_squared_error(yte, yhat)
    r2  = r2_score(yte, yhat)
    outer_scores.append([fold, mae, mse, r2, gs.best_params_])
    print(f"Fold {fold}: MAE={mae:.2f} | MSE={mse:.2f} | R²={r2:.3f} | best={gs.best_params_}")
    fold += 1

df_cvB = pd.DataFrame(outer_scores, columns=["fold","MAE","MSE","R2","best"])
print("\nNested CV (mean ± std):")
print(df_cvB[["MAE","MSE","R2"]].agg(['mean','std']))

# Split tunggal (agar apples-to-apples dengan Tes 2)
gss = GroupShuffleSplit(test_size=0.2, random_state=CFG["random_state"])
tr_idx, te_idx = next(gss.split(X_B, y_B, groups_B))
Xtr, Xte = X_B[tr_idx], X_B[te_idx]
ytr, yte = y_B[tr_idx], y_B[te_idx]

# Ambil best params dari rata-rata nested CV (atau pakai yang stabil)
best_params_B = {
    "n_estimators": 600,
    "max_depth": 12,
    "max_features": 0.5,
    "min_samples_leaf": 1,
    "random_state": CFG["random_state"],
    "n_jobs": -1
}
modelB = RandomForestRegressor(**best_params_B)
modelB.fit(Xtr, ytr)
yhat = modelB.predict(Xte)

mae = mean_absolute_error(yte, yhat)
mse = mean_squared_error(yte, yhat)
r2  = r2_score(yte, yhat)
print(f"\nTES 3 (Segmented) -> MAE={mae:.2f} | MSE={mse:.2f} | R²={r2:.3f}")

# CI bootstrap (rapi, langsung pada metrik)
def ci_bootstrap(y_true, y_pred, fn, n=1000, seed=42):
    rng = np.random.default_rng(seed); idx = np.arange(len(y_true)); vals=[]
    for _ in range(n):
        bs = rng.choice(idx, size=len(idx), replace=True)
        vals.append(fn(y_true[bs], y_pred[bs]))
    return np.percentile(vals, [2.5, 97.5])

mae_ci = ci_bootstrap(yte, yhat, lambda a,b: mean_absolute_error(a,b))
mse_ci = ci_bootstrap(yte, yhat, lambda a,b: mean_squared_error(a,b))
print(f"MAE 95% CI: [{mae_ci[0]:.2f}, {mae_ci[1]:.2f}]")
print(f"MSE 95% CI: [{mse_ci[0]:.2f}, {mse_ci[1]:.2f}]")


Fold 1: MAE=14.50 | MSE=400.34 | R²=0.755 | best={'max_depth': 12, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 600, 'n_jobs': -1, 'random_state': 42}
Fold 2: MAE=11.68 | MSE=380.36 | R²=0.744 | best={'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 3, 'n_estimators': 600, 'n_jobs': -1, 'random_state': 42}
Fold 3: MAE=11.60 | MSE=366.94 | R²=0.785 | best={'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 3, 'n_estimators': 600, 'n_jobs': -1, 'random_state': 42}
Fold 4: MAE=11.35 | MSE=285.51 | R²=0.830 | best={'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 300, 'n_jobs': -1, 'random_state': 42}
Fold 5: MAE=11.37 | MSE=342.98 | R²=0.777 | best={'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 600, 'n_jobs': -1, 'random_state': 42}

Nested CV (mean ± std):
            MAE         MSE        R2
mean  12.102296  355.224752  0.778337
std    1.349121   44.200637  0.033115

TES 3 (Segmented) -> M